# Dual Moving Average Crossover Strategy Backtest

**What this notebook does:**

Tests another classic Moving Average trading strategy:
- **BUY** (go long) when the stock price is **above** its moving average
- **GO TO CASH/SHORT** when the stock price is **below** its moving average

It then compares: did this strategy beat just holding the stock?

The second part brute-forces every fast/slow MA combination to find which pair would have performed best historically.

## Setup & Imports

4 Libraries
- `numpy` — fast math on arrays
- `yfinance` — free historical price data from Yahoo Finance
- `matplotlib` — plotting / charting
- `pandas` — DataFrames (spreadsheet-like tables for data manipulation)

In [ ]:
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd

## Configuration

Change these variables to test different assets or MA lengths:
- `TICKER` — which asset to backtest
- `FAST` — short-term moving average period (45 days)
- `SLOW` — long-term moving average period (200 days)
- `LOOKBACK` — only display the last x trading days for plotting

In [ ]:
TICKER = 'SPY'
FAST = 45
SLOW = 200
LOOKBACK = 1000

## Step 1: Download Price Data

Downloads the full history from Yahoo Finance, then trims to the last `LOOKBACK` rows for display.

The MAs are calculated *after* this slice, so the slow MA (x) won't have enough lookback for the first ~x rows — those will be `NaN`. That's expected and handled later.

In [ ]:
def get_data():
    df = yf.download(TICKER, start="1993-01-01")
    df.columns = df.columns.get_level_values(0)  # flatten multi-level column headers
    return df.iloc[-LOOKBACK:, :]                 # keep only the last LOOKBACK rows

## Step 2: Calculate & Plot the Two Moving Averages

Adds two MA columns and plots them alongside the closing price. The crossover points — where the fast MA crosses above or below the slow MA — are where the strategy generates its signals.

In [ ]:
def add_moving_averages(df, fast, slow):
    df[f'{FAST}_ma'] = df['Close'].rolling(fast).mean()
    df[f'{SLOW}_ma'] = df['Close'].rolling(slow).mean()

    plt.plot(df['Close'])
    plt.plot(df[f'{FAST}_ma'])
    plt.plot(df[f'{SLOW}_ma'])
    plt.legend(['Close', f'{FAST}_ma', f'{SLOW}_ma'])
    plt.title('Moving Average Crossovers');
    return df

## Step 3: Generate Trading Signals

The crossover rule:
- Fast MA **above** slow MA → `+1` (long) — momentum is bullish
- Fast MA **below** slow MA → `-1` (short) — momentum is bearish

### `.shift(1)` matters

Rows where either MA is `NaN` (not enough data yet) get set to `NaN` so we don't generate garbage signals. The `.shift(1)` avoids look-ahead bias.

In [ ]:
def add_strategy(df, fast, slow):
    df['Strategy'] = np.where(df[f'{fast}_ma'] > df[f'{slow}_ma'], 1, -1)

    # set to NaN where either MA doesn't have enough data
    df.loc[df[f'{fast}_ma'].isna() | df[f'{slow}_ma'].isna(), 'Strategy'] = np.nan

    df['Strategy'] = df['Strategy'].shift(1)  # avoid look-ahead bias
    return df

## Step 4: Calculate & Compare Returns

Computes cumulative returns for both buy-and-hold and the crossover strategy, then plots them.

**The math, step by step:**
1. `pct_change()` — calculates daily percentage change (e.g., 100 → 102 = +2%)
2. `1 + daily_return` — converts to a growth factor (e.g., +2% becomes 1.02)
3. `cumprod()` — multiplies all growth factors together cumulatively
4. `- 1` — converts back to percentage terms

When Strategy = **-1** (short), we multiply the daily return by -1. If the asset drops 1%, the strategy gains 1% and vice versa.

In [ ]:
def test_strategy(df, ticker, fast, slow):
    df['Asset_Returns'] = (1 + df['Close'].pct_change().fillna(0)).cumprod() - 1
    df['Strategy_Returns'] = (1 + (df['Close'].pct_change() * df['Strategy']).fillna(0)).cumprod() - 1

    plt.figure()
    plt.plot(df['Asset_Returns'])
    plt.plot(df['Strategy_Returns'])
    plt.legend([f'{ticker} Cumulative Returns', f'{fast} - {slow} Crossover Strategy Returns'])
    plt.title('Moving Average Crossovers')
    return df

## Step 5: Run the Backtest

In [ ]:
df = get_data()
df = add_moving_averages(df, FAST, SLOW)
df = add_strategy(df, FAST, SLOW)
df = test_strategy(df, TICKER, FAST, SLOW)

# drop rows where either MA was NaN (incomplete data at the start)
df = df.dropna(subset=[f'{FAST}_ma', f'{SLOW}_ma'])
df

---

Find the Best Fast/Slow MA Combination (Grid Search)

Tests a grid of fast/slow MA combinations and ranks them by final cumulative return.

In [ ]:
def find_best_ma_combo():
    results = []

    df_raw = yf.download(TICKER, start="1993-01-01")       # ← download once, before the loop
    df_raw.columns = df_raw.columns.get_level_values(0)

    for fast in range(5, 51, 5):
        for slow in range(20, 201, 10):
            if fast >= slow:
                continue

            df = df_raw.copy()                              # ← reuse a fresh copy

            df[f'{fast}_ma'] = df['Close'].rolling(fast).mean()
            df[f'{slow}_ma'] = df['Close'].rolling(slow).mean()

            strategy = np.where(df[f'{fast}_ma'] > df[f'{slow}_ma'], 1, -1)
            strategy = np.where(
                df[f'{fast}_ma'].isna() | df[f'{slow}_ma'].isna(),
                np.nan,
                strategy
            )

            strategy = np.roll(strategy, 1)  # shift by 1 to avoid look-ahead bias
            strategy[0] = np.nan             # first element is meaningless after roll

            returns = df['Close'].pct_change()
            strat_returns = (returns * strategy)
            strat_cum = (1 + strat_returns.fillna(0)).cumprod() - 1
            asset_cum = (1 + returns.fillna(0)).cumprod() - 1

            results.append({
                'fast': fast,
                'slow': slow,
                'strategy_final': strat_cum.iloc[-1],
                'asset_final': asset_cum.iloc[-1]
            })

    results = sorted(results, key=lambda x: x['strategy_final'], reverse=True)
    return results

results = find_best_ma_combo()

for r in results[:10]:
    print(r)